# GlyphLM Experiment

Does human-designed shorthand compression (stenography-inspired) make LLM tokenizers and training more efficient? This notebook runs the full pipeline: encode → tokenize → train → eval, comparing a tiny GPT-2 trained on raw `tiny_shakespeare` text against one trained on a glyph-compressed version.

Related work: [Training LLMs over Neurally Compressed Text](https://arxiv.org/abs/2404.03626) (Lester et al., 2024).

Runs end-to-end on Google Colab free tier (T4 GPU).

In [ ]:
!pip install -q torch transformers tokenizers datasets numpy

Clone the repo to get the `glyph` package. **Note:** if `shoegazerstella/glyph-lm` is private, an anonymous Colab runtime cannot clone it without a GitHub personal access token (`https://<token>@github.com/shoegazerstella/glyph-lm.git`), or you can make the repo public before running this notebook on Colab.

In [ ]:
!git clone https://github.com/shoegazerstella/glyph-lm.git
%cd glyph-lm

## Step 1: Shorthand Encoding

Applies deterministic regex rules (common words → glyphs, suffix collapsing, double-vowel dropping) to the raw corpus and measures the resulting whitespace-token compression ratio.

In [ ]:
from glyph import data

data.build_corpora()

## Step 2: Tokenizer Training

Trains two BPE tokenizers (vocab_size=2048) — one on raw text, one on glyph text — and reports vocab size, average tokens/line, and compression ratio vs. character count for each.

In [ ]:
from glyph import tokenizer as tok_mod

raw_tok = tok_mod.train_bpe("data/raw/train.txt", "tokenizers/raw_bpe")
glyph_tok = tok_mod.train_bpe("data/glyph/train.txt", "tokenizers/glyph_bpe")
tok_mod.report_stats(raw_tok, "data/raw/train.txt", "raw")
tok_mod.report_stats(glyph_tok, "data/glyph/train.txt", "glyph")

## Step 3: Model Training

Trains two identical tiny GPT-2 models (n_embd=128, n_layer=4, n_head=4) for 3 epochs each — one on raw text, one on glyph text — using MPS on Apple Silicon, CUDA on Colab's T4, or CPU as a fallback.

In [ ]:
from glyph import train

train.train_model("data/raw/train.txt", "tokenizers/raw_bpe", "models/model_raw", "raw")
train.train_model("data/glyph/train.txt", "tokenizers/glyph_bpe", "models/model_glyph", "glyph")

## Step 4: Evaluation

Compares both models on held-out perplexity, inference tokens/second, average tokens per line, and compression ratio — plus 3 sample completions each from the same seed prompt.

In [ ]:
from glyph import eval as eval_mod

eval_mod.main()

## Results & Interpretation

**Hypothesis recap:** _(fill in)_

**Observed results:**

| Metric | raw | glyph |
|---|---|---|
| Final train loss | | |
| Perplexity (val set) | | |
| Tokens/second (inference) | | |
| Avg tokens per line | | |
| Compression ratio | | |

**Discussion:** _(fill in)_

**Limitations:** _(fill in)_